In [1]:
import geopandas as gpd
import re
import pandas as pd
import numpy as np
import functools,operator
from pathlib import Path
from tqdm import tqdm
from matplotlib import pyplot as plt
from damagescanner.osm import read_osm_data

In [2]:
def convert_mixed_geometries_to_polygons(features, asset_type):
    """
    Convert point and linestring geometries to polygons for asset types with mixed geometry types.
    Only converts geometries for object types that have at least some polygon representations.
    Respects specific object types that should maintain their original geometry.
    
    Args:
        features (gpd.GeoDataFrame): Infrastructure features
        asset_type (str): Type of infrastructure asset
        
    Returns:
        gpd.GeoDataFrame: Features with consistent polygon geometries where appropriate
    """
    # Only apply for certain asset types
    if asset_type not in ['education', 'healthcare', 'telecom','power','gas', 'oil']:
        return features
    
    # Define object types that should NOT be converted for each asset type
    preserve_geometry = {
        'power': {
            'line': 'LineString',      # Should always remain as lines
            'tower': 'Point',          # Should always remain as points
            'pole': 'Point',           # Should always remain as points
            'catenary_mast': 'Point',  # Should always remain as points
            'cable': 'LineString',     # Should always remain as lines
            'minor_line': 'LineString' # Should always remain as lines
        },
        'gas': {
            'pipeline': 'LineString'   # Should always remain as lines
        },
        'oil': {
            'pipeline': 'LineString'   # Should always remain as lines
        },
        'telecom': {    
            'mast': 'Point',           # Should always remain as points
            'communications_tower': 'Point'  # Should always remain as points
        },
    }
    
    # Get the preserve list for this asset type
    preserve_list = preserve_geometry.get(asset_type, {})
    
    # Add geometry type information
    features['geom_type'] = features.geometry.geom_type
    
    # Create a mask for features that should preserve their geometry
    preserve_mask = pd.Series(False, index=features.index)
    for obj_type, geom_type in preserve_list.items():
        # Mark features with this object_type to preserve if they have the right geometry
        type_mask = (features['object_type'] == obj_type) & (features['geom_type'] == geom_type)
        preserve_mask = preserve_mask | type_mask
    
    # Get polygon features to calculate median areas (only for non-preserved features)
    polygon_features = features.loc[
        (~preserve_mask) & features.geom_type.isin(['Polygon', 'MultiPolygon'])
    ].to_crs(3035)
    
    # If no polygon features exist, return original features
    if len(polygon_features) == 0:
        features = features.drop(['geom_type'], axis=1)
        return features
        
    polygon_features['square_m2'] = polygon_features.area
    
    # Calculate median area by object type
    square_m2_object_type = polygon_features[['object_type', 'square_m2']].groupby('object_type').median()
    
    # Default area if median cannot be calculated (1000 sq meters ~ small building)
    default_area = 1000
    
    # Find object types that have mixed geometries (linestrings + polygons)
    # Only consider non-preserved features
    non_preserved_features = features[~preserve_mask]
    mixed_geom_types = non_preserved_features.groupby(['object_type', 'geom_type']).size().unstack().fillna(0)
    
    # Identify object types that have both linestrings and polygons
    linestrings_to_polygonize = []
    if 'LineString' in mixed_geom_types.columns and any(col in mixed_geom_types.columns for col in ['Polygon', 'MultiPolygon']):
        for obj_type in mixed_geom_types.index:
            # Skip if this object type should be preserved
            if obj_type in preserve_list and preserve_list[obj_type] == 'LineString':
                continue
                
            line_count = mixed_geom_types.loc[obj_type, 'LineString'] if 'LineString' in mixed_geom_types.columns else 0
            poly_count = sum(mixed_geom_types.loc[obj_type, col] for col in ['Polygon', 'MultiPolygon'] 
                            if col in mixed_geom_types.columns)
            
            # If this object type has both linestrings and polygons, add to conversion list
            if line_count > 0 and poly_count > 0:
                linestrings_to_polygonize.append(obj_type)
    
    # Convert linestrings to polygons
    if linestrings_to_polygonize:
        print(f"Converting linestrings to polygons for {asset_type}: {linestrings_to_polygonize}")
        
        # Get linestrings to convert
        all_linestrings_to_polygonize = features.loc[
            (features.object_type.isin(linestrings_to_polygonize)) & 
            (features.geom_type == 'LineString') &
            (~preserve_mask)  # Ensure we don't convert preserved features
        ]
        
        if len(all_linestrings_to_polygonize) > 0:
            # Define function to convert linestring to polygon
            def polygonize_linestring(linestring):
                try:
                    # Simple conversion for closed linestrings
                    if linestring.is_closed:
                        return shapely.geometry.Polygon(linestring)
                    else:
                        # For open linestrings, create a small buffer
                        return linestring.buffer(0.0001)
                except Exception:
                    # Fallback: create a small buffer
                    return linestring.buffer(0.0001)
            
            # Apply conversion
            new_geometries = all_linestrings_to_polygonize.geometry.apply(polygonize_linestring).values
            
            # Update geometries
            features.loc[
                (features.object_type.isin(linestrings_to_polygonize)) & 
                (features.geom_type == 'LineString') &
                (~preserve_mask),  # Ensure we don't convert preserved features
                'geometry'
            ] = new_geometries
    
    # Get the points to convert (only for object types that also have polygons)
    points_to_polygonize = []
    if 'Point' in mixed_geom_types.columns and any(col in mixed_geom_types.columns for col in ['Polygon', 'MultiPolygon']):
        for obj_type in mixed_geom_types.index:
            # Skip if this object type should be preserved
            if obj_type in preserve_list and preserve_list[obj_type] == 'Point':
                continue
                
            point_count = mixed_geom_types.loc[obj_type, 'Point'] if 'Point' in mixed_geom_types.columns else 0
            poly_count = sum(mixed_geom_types.loc[obj_type, col] for col in ['Polygon', 'MultiPolygon'] 
                            if col in mixed_geom_types.columns)
            
            # If this object type has both points and polygons, add to conversion list
            if point_count > 0 and poly_count > 0:
                points_to_polygonize.append(obj_type)
    
    if points_to_polygonize:
        all_assets_to_polygonize = features.loc[
            (features.object_type.isin(points_to_polygonize)) & 
            (features.geom_type == 'Point') &
            (~preserve_mask)  # Ensure we don't convert preserved features
        ].to_crs(3035)
        
        if len(all_assets_to_polygonize) > 0:
            print(f"Converting {len(all_assets_to_polygonize)} points to polygons for {asset_type}: {points_to_polygonize}")
            
            # Define function to polygonize points
            def polygonize_point_per_asset(asset):
                # Get buffer length (half of width/length)
                if asset.object_type in square_m2_object_type.index:
                    area = square_m2_object_type.loc[asset.object_type].values[0]
                else:
                    area = default_area
                    
                buffer_length = np.sqrt(area) / 2
                
                # Buffer the point to create a square polygon
                return asset.geometry.buffer(buffer_length, cap_style='square')
            
            # Apply the conversion
            new_geometries = all_assets_to_polygonize.apply(
                lambda asset: polygonize_point_per_asset(asset), axis=1
            ).set_crs(3035).to_crs(3035).values
            
            # Update the geometries
            features.loc[
                (features.object_type.isin(points_to_polygonize)) & 
                (features.geom_type == 'Point') &
                (~preserve_mask),  # Ensure we don't convert preserved features
                'geometry'
            ] = new_geometries
    
    # Remove the temporary geom_type column
    features = features.drop(['geom_type'], axis=1)
    
    return features


In [3]:
country = 'ESP'

In [4]:
data_path_TENT = Path(r"C:\Users\eks510\OneDrive - Vrije Universiteit Amsterdam\Documenten - MIRACA\General\TENT-data layers")
LAU_path = r"C:\Users\eks510\OneDrive - Vrije Universiteit Amsterdam\Documenten - MIRACA\General\Deliverables and Milestones\Milestone_Intermediate version of harmonised exposure database\Data\LAU_RG_01M_2024_3035.geojson"
NUTS_path = r"C:\Users\eks510\OneDrive - Vrije Universiteit Amsterdam\Documenten - MIRACA\General\Deliverables and Milestones\Milestone_Intermediate version of harmonised exposure database\Data\NUTS_RG_01M_2024_3035.geojson"

In [5]:
%%time
roads = gpd.read_parquet(data_path_TENT / 'europe_road_edges_TENT.parquet').to_crs(3035)
rail = gpd.read_parquet(data_path_TENT / 'europe_railways_edges_TENT.parquet')

CPU times: total: 1min 3s
Wall time: 1min 9s


array(['trunk', 'secondary', 'primary', 'tertiary', 'motorway'],
      dtype=object)

In [10]:
corridor_roads = roads[(roads['CORRIDORS'].notna()) & (roads['CORRIDORS'] != 'nan')]

In [27]:
corridor_roads['object_type'] = corridor_roads.tag_highway

C:\Users\eks510\.conda\envs\pygis\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [28]:
corridor_roads

,geometry,osm_way_id,segment_id,tag_highway,tag_surface,tag_bridge,tag_maxspeed,tag_lanes,bridge,id,...,from_iso_a3,to_iso_a3,paved,material,lanes,asset_type,component_id,CORRIDORS,GIS_STATUS,object_type
27040,"LINESTRING (2694705.009 1984888.212, 2694761.5...",4.015552e+06,0.0,motorway,None,None,NaN,1.0,False,europe-latest_17_7,...,PRT,PRT,True,asphalt,1.0,road_motorway,1,G,Completed,motorway
27081,"LINESTRING (2924835.282 1598057.169, 2924771.6...",4.060050e+06,0.0,motorway,asphalt,None,120.0,2.0,False,europe-latest_17_48,...,ESP,ESP,True,asphalt,2.0,road_motorway,1,CG,Completed,motorway
27084,"LINESTRING (2904284.101 1635311.743, 2904388.9...",4.060076e+06,0.0,motorway,asphalt,None,100.0,2.0,False,europe-latest_17_51,...,ESP,ESP,True,asphalt,2.0,road_motorway,1,CG,Completed,motorway
27087,"LINESTRING (2706565.691 1991214.586, 2706385.2...",4.246625e+06,0.0,motorway,asphalt,None,120.0,3.0,False,europe-latest_17_54,...,PRT,PRT,True,asphalt,3.0,road_motorway,1,G,Completed,motorway
27088,"LINESTRING (2677599.789 1963518.531, 2677830.1...",4.246626e+06,0.0,motorway,asphalt,None,120.0,3.0,False,europe-latest_17_55,...,PRT,PRT,True,asphalt,3.0,road_motorway,1,G,Completed,motorway
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9722689,"LINESTRING (6352883.841 3047917.322, 6352909.1...",1.026093e+09,0.0,trunk,asphalt,None,NaN,2.0,False,europe-latest_59_53480,...,UKR,UKR,True,asphalt,2.0,road_trunk,1,B,Completed,trunk
9722690,"LINESTRING (6353459.593 3048413.726, 6353601.6...",1.026093e+09,1.0,trunk,asphalt,None,NaN,2.0,False,europe-latest_59_53481,...,UKR,UKR,True,asphalt,2.0,road_trunk,1,B,Completed,trunk
9722707,"LINESTRING (6342479.677 3039618.136, 6342526.0...",1.026991e+09,0.0,primary,asphalt,None,90.0,2.0,False,europe-latest_59_53498,...,UKR,UKR,True,asphalt,2.0,road_primary,1,B,Completed,primary
9722708,"LINESTRING (6344847.854 3041073.294, 6344873.4...",1.026991e+09,0.0,primary,asphalt,None,50.0,2.0,False,europe-latest_59_53499,...,UKR,UKR,True,asphalt,2.0,road_primary,1,B,Completed,primary


In [32]:
corridor_roads.reset_index(drop=True).to_parquet(r"C:\MIRACA\infrastructure_data\roads_tent.parquet")

In [36]:
corridor_roads

,geometry,osm_way_id,segment_id,tag_highway,tag_surface,tag_bridge,tag_maxspeed,tag_lanes,bridge,id,...,from_iso_a3,to_iso_a3,paved,material,lanes,asset_type,component_id,CORRIDORS,GIS_STATUS,object_type
27040,"LINESTRING (2694705.009 1984888.212, 2694761.5...",4.015552e+06,0.0,motorway,None,None,NaN,1.0,False,europe-latest_17_7,...,PRT,PRT,True,asphalt,1.0,road_motorway,1,G,Completed,motorway
27081,"LINESTRING (2924835.282 1598057.169, 2924771.6...",4.060050e+06,0.0,motorway,asphalt,None,120.0,2.0,False,europe-latest_17_48,...,ESP,ESP,True,asphalt,2.0,road_motorway,1,CG,Completed,motorway
27084,"LINESTRING (2904284.101 1635311.743, 2904388.9...",4.060076e+06,0.0,motorway,asphalt,None,100.0,2.0,False,europe-latest_17_51,...,ESP,ESP,True,asphalt,2.0,road_motorway,1,CG,Completed,motorway
27087,"LINESTRING (2706565.691 1991214.586, 2706385.2...",4.246625e+06,0.0,motorway,asphalt,None,120.0,3.0,False,europe-latest_17_54,...,PRT,PRT,True,asphalt,3.0,road_motorway,1,G,Completed,motorway
27088,"LINESTRING (2677599.789 1963518.531, 2677830.1...",4.246626e+06,0.0,motorway,asphalt,None,120.0,3.0,False,europe-latest_17_55,...,PRT,PRT,True,asphalt,3.0,road_motorway,1,G,Completed,motorway
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9722689,"LINESTRING (6352883.841 3047917.322, 6352909.1...",1.026093e+09,0.0,trunk,asphalt,None,NaN,2.0,False,europe-latest_59_53480,...,UKR,UKR,True,asphalt,2.0,road_trunk,1,B,Completed,trunk
9722690,"LINESTRING (6353459.593 3048413.726, 6353601.6...",1.026093e+09,1.0,trunk,asphalt,None,NaN,2.0,False,europe-latest_59_53481,...,UKR,UKR,True,asphalt,2.0,road_trunk,1,B,Completed,trunk
9722707,"LINESTRING (6342479.677 3039618.136, 6342526.0...",1.026991e+09,0.0,primary,asphalt,None,90.0,2.0,False,europe-latest_59_53498,...,UKR,UKR,True,asphalt,2.0,road_primary,1,B,Completed,primary
9722708,"LINESTRING (6344847.854 3041073.294, 6344873.4...",1.026991e+09,0.0,primary,asphalt,None,50.0,2.0,False,europe-latest_59_53499,...,UKR,UKR,True,asphalt,2.0,road_primary,1,B,Completed,primary


In [37]:
corridor_roads.groupby(['id','CORRIDORS']).count()

,,geometry,osm_way_id,segment_id,tag_highway,tag_surface,tag_bridge,tag_maxspeed,tag_lanes,bridge,from_id,to_id,from_iso_a3,to_iso_a3,paved,material,lanes,asset_type,component_id,GIS_STATUS,object_type
id,CORRIDORS,,,,,,,,,,,,,,,,,,,,
europe-latest_17_10000,G,1,1,1,1,1,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1
europe-latest_17_10001,G,1,1,1,1,1,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1
europe-latest_17_100016,CG,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1
europe-latest_17_100017,CG,1,1,1,1,1,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1
europe-latest_17_100018,CG,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
europe-latest_59_9947,B,1,1,1,1,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1
europe-latest_59_9949,B,1,1,1,1,1,0,1,0,1,1,1,1,1,1,1,1,1,1,1,1
europe-latest_59_9950,B,1,1,1,1,1,0,1,0,1,1,1,1,1,1,1,1,1,1,1,1


In [14]:
corridor_rail = rail[(rail['CORRIDORS'].notna()) & (rail['CORRIDORS'] != 'nan')]

In [29]:
corridor_rail['object_type'] = 'rail'

C:\Users\eks510\.conda\envs\pygis\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [31]:
corridor_rail.reset_index(drop=True).to_parquet(r"C:\MIRACA\infrastructure_data\rail_tent.parquet")

In [6]:
asset_type = 'power'

In [7]:
%%time
osm_path = Path(r"C:\Data\country_osm\spain-250922.osm.pbf")
features = read_osm_data(osm_path,asset_type=asset_type)
features_index = features.sindex

C:\Users\eks510\.conda\envs\pygis\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: Non closed ring detected. To avoid accepting it, set the OGR_GEOMETRY_ACCEPT_UNCLOSED_RING configuration option to NO
  return ogr_read(


CPU times: total: 9min 49s
Wall time: 21min 32s


In [8]:
%%time
LAU = gpd.read_file(LAU_path, engine="pyogrio")
NUTS = gpd.read_file(NUTS_path, engine="pyogrio")

CPU times: total: 18 s
Wall time: 30.4 s


In [9]:
NUTS2 = NUTS.loc[NUTS.LEVL_CODE == 2].reset_index(drop=True)
NUTS_index = NUTS2.sindex

In [10]:
def get_NUTS2(LAU_region,NUTS2):

    NUTS2_rough = NUTS2.iloc[NUTS_index.intersection((LAU_region.geometry.centroid.x,LAU_region.geometry.centroid.y))]
    boolean = NUTS2_rough.intersects(LAU_region.geometry.centroid).values 
    try:
        return NUTS2_rough.loc[boolean].NUTS_ID.values[0]
    except:
        return None

In [11]:
LAU_country = LAU.loc[LAU.CNTR_CODE == 'ES']

In [12]:
tqdm.pandas()
LAU_country.loc[:,'NUTS2'] = LAU_country.progress_apply(lambda LAU_region: 
                                              get_NUTS2(LAU_region,NUTS2),axis=1)

100%|████████████████████████████████████████████████████████████████████████████| 8132/8132 [00:06<00:00, 1216.37it/s]
C:\Users\eks510\.conda\envs\pygis\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [13]:
collect_assets = []
for LAU_region in tqdm(LAU_country.to_crs(4326).itertuples(), total=len(LAU_country)):
    # Get rough intersection using spatial index
    assets_rough = features.iloc[features_index.intersection(LAU_region.geometry.bounds)].copy()
    
    # Skip if no assets found in bounding box
    if assets_rough.empty:
        continue
        
    # Calculate precise intersection - assign to geometry column correctly
    assets_rough['geometry'] = assets_rough.geometry.intersection(LAU_region.geometry)
    
    # Filter out empty geometries and make a copy to avoid warnings
    assets_precise = assets_rough[~assets_rough.geometry.is_empty].copy()
    
    # Now safely assign new columns
    assets_precise['LAU'] = LAU_region.GISCO_ID
    assets_precise['NUTS2'] = LAU_region.NUTS2
    assets_precise['CNTR_CODE'] = LAU_region.CNTR_CODE
    collect_assets.append(assets_precise)
  
country_assets = pd.concat(collect_assets).to_crs(3035)
country_assets.osm_id = country_assets.osm_id.astype(np.float64)
country_assets = country_assets.loc[country_assets.is_valid]
country_assets = country_assets.reset_index(drop=True)
country_assets = convert_mixed_geometries_to_polygons(country_assets, asset_type)


100%|█████████████████████████████████████████████████████████████████████████████| 8132/8132 [00:25<00:00, 320.12it/s]


Converting linestrings to polygons for power: ['generator', 'plant', 'substation', 'terminal', 'tower', 'transformer']
Converting 36751 points to polygons for power: ['generator', 'plant', 'substation', 'switch', 'terminal', 'transformer']


In [16]:
country_assets

,osm_id,geometry,object_type,voltage,utility,name,source,LAU,NUTS2,CNTR_CODE
0,3.773358e+08,"LINESTRING (3172396.07 1662364.419, 3172822.03...",minor_line,None,None,None,None,ES_04045,ES61,ES
1,3.957067e+09,POINT (3172822.037 1662628.387),pole,None,None,None,None,ES_04045,ES61,ES
2,3.957067e+09,POINT (3172968.409 1662713.204),pole,None,None,None,None,ES_04045,ES61,ES
3,3.957123e+09,"POLYGON ((3172616.609 1662953.032, 3172616.609...",transformer,None,None,None,None,ES_04045,ES61,ES
4,3.957123e+09,POINT (3172585.117 1662956.393),pole,None,None,None,None,ES_04045,ES61,ES
...,...,...,...,...,...,...,...,...,...,...
654882,1.013545e+10,POINT (3339553.215 2001211.704),tower,None,None,None,None,ES_44235,ES24,ES
654883,1.013545e+10,POINT (3339256.867 2001303.299),tower,None,None,None,None,ES_44235,ES24,ES
654884,1.013545e+10,POINT (3339113.217 2001347.6),tower,None,None,None,None,ES_44235,ES24,ES
654885,1.013545e+10,POINT (3338841.911 2001422.1),tower,None,None,None,None,ES_44235,ES24,ES


In [15]:
country_assets.to_parquet(f"C:\\MIRACA\\infrastructure_data\\{country}_{asset_type}.parquet")

In [12]:
#corridor_dict = dict(zip(roads.osm_way_id.values,roads.CORRIDORS.values))
corridor_dict = dict(zip(rail.osm_way_id.values,rail.CORRIDORS.values))

In [13]:
def corridor_mapping(osm_id,corridor_dict):
    try:
        return corridor_dict[osm_id]
    except:
        return None

tqdm.pandas()
country_roads['CORRIDOR'] = country_roads.osm_id.progress_apply(lambda osm_id : corridor_mapping(osm_id,corridor_dict))

100%|██████████████████████████████████████████████████████████████████████████| 9404/9404 [00:00<00:00, 415883.63it/s]
